Scaling exponent part for logistic map

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import networkx as nx
import math

Constructing the RC skeleton

In [ ]:
class ReservoirComputer:
    
    def __init__(self, dim_system, dim_reservoir, rho, sigma):
        
        self.dim_system = dim_system
        self.dim_reservoir = dim_reservoir
        self.r_state = np.zeros(dim_reservoir)                                   
        self.A = generate_adjacency_matrix(dim_reservoir, rho, sigma)

        
        self.W_in = W_inn(dim_reservoir, dim_system)
        self.W_b = W_bb(dim_reservoir)
        self.W_out = np.zeros((dim_system, dim_reservoir))
        
        

    
    def advance_r_state(self, eps, u):
        alpha = 0.86
        self.r_state = ((1- alpha)* self.r_state) + alpha* np.tanh(np.matmul(self.A, self.r_state) + np.matmul(self.W_in, u.T) + self.W_b * (eps - 0.3))             # r_state is updated and stored as r(t+ delta t)
                                                                                    
        return self.r_state

    
    def train(self, trajectory_total, trajectory_new):                                                # Trajectory is the training data (initialized once model is called)
        
    
        R_total = np.empty((self.dim_reservoir, trajectory_total.shape[0]))
        eps_train = [3.92, 3.93, 3.94, 3.95]
                                                                                                      
        div = int((trajectory_total.shape[0])/4)
        
        for m in range(len(eps_train)):

            self.r_state = np.zeros(self.dim_reservoir)
            
            for i in range(div*m , div + div*m):
                
                R_total[:, i] = self.r_state
                u = trajectory_total[i]                                                   
                self.advance_r_state(eps_train[m], u)

        range_removal = [(0,29),(1000,1029),(2000, 2029),(3000, 3029)]
        column_removal = []
        for start, end in range_removal:
            column_removal.extend(range(start, end + 1))
        
        # Remove the specified columns
        R_total = np.delete(R_total, column_removal, axis=1)      #removing the r_state transient (first 10 elements)         
        
        odd_column_indices = np.arange(R_total.shape[1])[1::2]    #can be commented out as logistic map doesn't have a symmetry problem (that will only change the crisis point slightly in trained RC, since W_out changes)                                                                 

        R_total[:, odd_column_indices] **= 2                      #need to keep only if the exact reproduction of parameter values given in the paper at which critical transition occurs, is desired
        
        
        self.W_out = linear_regression(R_total, trajectory_new)

        

    def v(self):
        return np.transpose(np.matmul(self.W_out, self.r_state))                   # Gives the predicted output


    def predict(self, eps_new, steps, test_trajectory, train_trajectory):
        
        prediction = np.zeros((steps, self.dim_system))
        prediction[0] = test_trajectory[0,:]

        warmup = train_trajectory[0:30, :]                       # warmup of reservoir

        self.r_state = np.zeros(self.dim_reservoir)

        for l in range(30):

            self.advance_r_state(eps_new, warmup[l])

        self.r_warmup = self.r_state
        
        for i in range(steps-1):
            
            self.advance_r_state(eps_new, prediction[i])
            v = self.v()
            prediction[i+1] = v
            
        return prediction

    def get_weights(self):
        return self.A, self.W_in, self.W_out, self.W_b, self.r_warmup

In [ ]:
def W_bb(dim_reservoir):

    np.random.seed(36)
    
    W_b = (2 * np.random.rand(dim_reservoir) - 1)* 1.15 

    return W_b


def W_inn(dim_reservoir, dim_system):

    np.random.seed(85)
    W_in = 2 * 2.13 * (np.random.rand(dim_reservoir, dim_system) - 0.5)
    
    return W_in

def generate_adjacency_matrix(dim_reservoir, rho, sigma):
    
    graph = nx.gnp_random_graph(dim_reservoir, sigma, seed = 47)                           # Generates an Erdos-Renyi graph of nodes = dim_reservoir and probab of node connection = sigma
    
    np.random.seed(27)
    nx.set_edge_attributes(graph, {e: {'weight': np.random.uniform(0, 1)} for e in graph.edges})    
    graph = nx.to_numpy_array(graph)

    
    rescaled = graph
    return scale_matrix(rescaled, rho)


def scale_matrix(A, rho):
    eigenvalues, _ = np.linalg.eig(A)
    max_eigenvalue = np.amax(eigenvalues)
    A = (A/np.absolute(max_eigenvalue)) * rho                 
    return A


In [ ]:
def linear_regression(R, trajectory, beta=0.000001):
    Rt = np.transpose(R)
    inverse_part = np.linalg.inv(np.matmul(R, Rt) + beta * np.identity(R.shape[0]))                
    return np.matmul(np.matmul(trajectory.T, Rt), inverse_part)                          

Constructing the data (logistic map time series)

In [ ]:
timesteps = 1100                                                #Trained for 1000 timesteps and removed 100 as transient

mu_list = [3.92, 3.93, 3.94, 3.95, 4.01]      

data_list = []

# Initial conditions
for mu in mu_list:
    x0 = 0.3
    for j in range(timesteps):

        x0 = mu*x0*(1-x0)
   
        data_list.append(x0)
   
stacked_total = np.transpose(np.vstack(data_list))                                          
print(stacked_total.shape)

In [ ]:
#train-test split

stacked_train = stacked_total[:, :-1100]
stacked_test = stacked_total[:, -1100:]
stacked_test = stacked_test[:, :1000]       

# removing transient of first 100 points each time
ranges_to_remove = [(0, 99), (1100, 1199), (2200, 2299), (3300,3399)]
columns_to_remove = []
for start, end in ranges_to_remove:
    columns_to_remove.extend(range(start, end + 1))


stacked_x_modified = np.delete(stacked_train, columns_to_remove, axis=1)
                 
training_data = np.transpose(stacked_x_modified)                

# Remove the reservoir transient, 30 points each time
range_to_remove = [(0,29),(1000,1029),(2000, 2029),(3000, 3029)]

column_to_remove = []
for start, end in range_to_remove:
    column_to_remove.extend(range(start, end + 1))

# Remove the specified columns
train_without_transient = np.transpose(np.delete(stacked_x_modified, column_to_remove, axis=1))            

valid_data = np.transpose(stacked_test)              


Checking the RC prediction

In [ ]:
dim_reservoir = 400                                    
model = ReservoirComputer(1, dim_reservoir, 0.9, 0.526)                       

model.train(training_data, train_without_transient)
predicted_data = model.predict(4 ,len(valid_data), valid_data, train_without_transient)              

In [ ]:
#checking the plots
x_data = valid_data[:,0]

x_pred = predicted_data[:,0]

timeaxis = np.arange(0, (valid_data.shape[0]), 1)

h = 20
l = 200
plt.plot(timeaxis[:h], x_data[:h], label = "x")
plt.show()
plt.plot(timeaxis[:l], x_pred[:l], label = "x_pred")
plt.show()


Predicted time series using the trained RC map

In [ ]:
A, W_in, W_out, W_b, rstate_initial = model.get_weights()

Lambda = A + np.matmul(W_in, W_out)
Omega = W_b


In [ ]:

rstate_initial1 = rstate_initial.copy()
alpha = 0.86
bif_param = 3.999
n = 700
x_array = np.empty((1, n))
time = np.arange(0,n,1)
predd_data = np.matmul(W_out, rstate_initial1)

for j in range(n):
    r = (1-alpha)*rstate_initial1 + alpha* np.tanh(np.matmul(A, rstate_initial1) + np.matmul(W_in, predd_data) + Omega*(bif_param - 0.3))
    rstate_initial1 = r
    predd_data = np.matmul(W_out, r)
    x_array[:,j] = predd_data

plt.plot(time, x_array[0, :],
         color='darkgreen',  
         linewidth=1.2)

plt.xlabel("t")
plt.ylabel("x")

plt.tight_layout()
plt.show()

Plotting density function of trained map

In [ ]:

# Parameters
bifparam = 3.9985
alpha = 0.86
timesteps = 50000

# Get model weights
A, W_in, W_out, W_b, rstate_initial = model.get_weights()
Lambda = A + np.matmul(W_in, W_out)
Omega = W_b

pred_store = []
r_sstate = np.zeros(400)

# Warmup with logistic map
x0_list = []
x0 = np.random.uniform(0.2, 0.6)
for _ in range(300):
    x0 = 3.99 * x0 * (1 - x0)
    x0_list.append(x0)

x0_array = np.array(x0_list).reshape(300, 1)
warmup_array = x0_array[-41:]

# Warmup phase
for l in range(40):
    rstate_initial1 = (1 - alpha) * r_sstate + alpha * np.tanh(
        np.matmul(A, r_sstate) +
        np.matmul(W_in, warmup_array[l, :].T) +
        W_b * (bifparam - 0.3)
    )
    r_sstate = rstate_initial1

preddd_data = warmup_array[-1, :]

# Simulation
for k in range(timesteps):
    r = (1 - alpha) * rstate_initial1 + alpha * np.tanh(
        np.matmul(A, rstate_initial1) +
        np.matmul(W_in, preddd_data) +
        Omega * (bifparam - 0.3)
    )
    rstate_initial1 = r
    preddd_data = np.matmul(W_out, r)
    pred_store.append(preddd_data)

pred_store_array = np.array(pred_store)

plt.hist(pred_store_array, bins=100, density=True,
         color='darkgreen', alpha=0.85, edgecolor='white', linewidth=0.3)

plt.xlabel(r"$x$")
plt.ylabel(r"$\rho$")

plt.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()

plt.show()



Checking the return maps

In [ ]:
epsilon_diff = 0.0001

og_bif = 4 + epsilon_diff
res_bif = 3.998548 + epsilon_diff

m_log = 200
m_res = 200
# Return map of original logistic equation

initial_cond = 0.3
logistic_list = []
res_list = []

for i in range(m_log):
    initial_cond = og_bif *(initial_cond)*(1 - initial_cond)
    logistic_list.append(initial_cond)

# Return map of reservoir map

A, W_in, W_out, W_b, rstate_initial = model.get_weights()

Lambda = A + np.matmul(W_in, W_out)
Omega = W_b

alpha = 0.86

r_sstate = np.zeros(400)
x0_list = []

x0 = 0.3                                            #check for any initial condition, you should see similar behaviour
for m in range(200):

    x0 = 3.99*x0*(1 - x0)
    x0_list.append(x0)

x0_array = np.array(x0_list).reshape(200, 1)
warmup_array = x0_array[-41:]  
    
y_init = warmup_array[-1,:]

for l in range(40):

    rstate_initial1 = ((1- alpha)* r_sstate) + alpha* np.tanh(np.matmul(A, r_sstate) + np.matmul(W_in, warmup_array[l,:].T) + W_b * (res_bif - 0.3))
    r_sstate = rstate_initial1


def h(r_sstate, x):

    r = (1-alpha)*r_sstate + alpha* np.tanh(np.matmul(A, r_sstate) + np.matmul(W_in, x)+ Omega*(res_bif - 0.3))
    
    y = np.matmul(W_out, r)                                                                                                                
                                                                                                                                    
    return y,r


for j in range(m_res):
    y1, rstate_initiall = h(r_sstate, y_init)
    res_list.append(y_init)
    y_init = y1
    r_sstate = rstate_initiall

In [ ]:
logistic_array = np.array(logistic_list)
res_array = np.array(res_list)

# Prepare return map coordinates: (x_n, x_{n+1})
x_log = logistic_array[:-1]
y_log = logistic_array[1:]

x_res = res_array[:-1]
y_res = res_array[1:]

plt.scatter(x_log, y_log, color='blue', alpha=0.6,
            label='Logistic Map', s=25)  # was s=10

plt.scatter(x_res, y_res, marker='x', color='red', alpha=0.6,
            label='Reservoir Map', s=80, linewidth=1.5)  # was s=10

# Square box and diagonal
plt.plot([0, 1], [0, 1], ls='--', color='green', linewidth=1.5, label='y = x')  
plt.plot([0, 0, 1, 1, 0], [0, 1, 1, 0, 0], ls='--', color='green', linewidth=1.5)  

plt.xlabel(r'$x_n$')
plt.ylabel(r'$x_{n+1}$')

plt.grid(True)
plt.gca().set_aspect('equal', adjustable='box')  # Make it square
plt.tight_layout()

plt.show()


Scaling exponent logistic map

In [ ]:
bif_array_logmap = np.array([4.0005, 4.001, 4.005, 4.01, 4.05, 4.1 ])

ttimesteps = 500
xx_initial = 0.5
cc = 0.3
nnum_trials = 50000
tthreshold = 0


def simulate_single_perturb_logmap(bif_param):
    x_init = xx_initial + np.random.uniform(-cc, cc)
    
    for k in range(ttimesteps):
        x_init = bif_param * x_init * (1 - x_init)  # logistic map rule
        if x_init < tthreshold:                      # break early
            return k                                # store crossing point

    return ttimesteps  # only reached if condition never met


scaling_list_avg_logmap = []
for bif_param in bif_array_logmap:

    rresults = []
    for p in range(nnum_trials):
        rresult = simulate_single_perturb_logmap(bif_param)
        rresults.append(rresult)
    scaling_list_avg_logmap.append(np.mean(rresults))

    print(scaling_list_avg_logmap)


bif_array_shift_logmap = bif_array_logmap - 4
slope = (np.log(scaling_list_avg[-2]) - np.log(scaling_list_avg[-1]))/(np.log(bif_array_shift[-2]) - np.log(bif_array_shift[-1]))


plt.loglog(bif_array_shift_logmap, scaling_list_avg_logmap, marker='o', linestyle='-')
plt.xlabel("Bifurcation parameter - Kc")
plt.ylabel("Average steps before collapse")
plt.grid(True, which='both')
plt.show()

Scaling exponent RC map

In [ ]:
bif_array = np.array([3.999,  4.0001, 4.0005, 4.001, 4.005, 4.01, 4.05, 4.1])

timesteps = 200000
alpha = 0.86
num_trials = 10000
threshold = 0             # Collapse threshold
exit_threshold = 1.5         # Divergence threshold

scaling_list_avg = []

A, W_in, W_out, W_b, rstate_initial = model.get_weights()
Lambda = A + np.matmul(W_in, W_out)
Omega = W_b

# Simulation function
def simulate_single_perturb(bif_param):
    r_sstate = np.zeros(400)

    # Generate chaotic warmup input from logistic map
    x0_list = []
    x0 = np.random.uniform(0.2, 0.6)
    for _ in range(300):
        x0 = 3.99 * x0 * (1 - x0)
        x0_list.append(x0)

    x0_array = np.array(x0_list).reshape(300, 1)
    warmup_array = x0_array[-101:]  # Shape: (41, 1)

    # Warmup phase
    for l in range(100):
        rstate_initial1 = (1 - alpha) * r_sstate + alpha * np.tanh(
            np.matmul(A, r_sstate) +
            np.matmul(W_in, warmup_array[l, :].T) +
            W_b * (bif_param - 0.3)
        )
        r_sstate = rstate_initial1

    preddd_data = warmup_array[-1, :]

    # Simulation loop
    for k in range(timesteps):
        r = (1 - alpha) * rstate_initial1 + alpha * np.tanh(
            np.matmul(A, rstate_initial1) +
            np.matmul(W_in, preddd_data) +
            Omega * (bif_param - 0.3)
        )
        rstate_initial1 = r
        preddd_data = np.matmul(W_out, r)

        val = preddd_data.item()  # Convert to scalar

        if val < threshold:
            return k              # Collapse
        elif val > exit_threshold:
            return None           # Divergence

    return None              # No collapse or divergence

from tqdm import tqdm

for i, bif_param in enumerate(bif_array):
    print(f"\nRunning for bif_param = {bif_param:.5f} ({i + 1}/{len(bif_array)})...")
    results = []

    for _ in tqdm(range(num_trials), desc=f"Trials for bif={bif_param:.5f}"):
        result = simulate_single_perturb(bif_param)
        if result is not None:
            results.append(result)

    skipped = num_trials - len(results)

    if results:
        avg = np.mean(results)
    else:
        avg = float('nan')

    scaling_list_avg.append(avg)

    print(f"  Average time to collapse (valid trials = {len(results)}): {avg:.2f}")
    print(f"  Skipped {skipped} divergent trials")





In [ ]:
bif_array_shift = bif_array - 3.998548

log_x = np.log(bif_array_shift)
log_y = np.log(scaling_list_avg)
slope, _ = np.polyfit(log_x, log_y, 1)
print(f"Estimated scaling exponent γ ≈ {-slope:.3f}")

# Plot
plt.loglog(bif_array_shift, scaling_list_avg, marker='o', linestyle='-')
plt.xlabel("Bifurcation parameter - Kc")
plt.ylabel("Average steps before collapse")
plt.grid(True, which='both')
plt.show()

Comparison of scaling exponents

In [ ]:

from scipy.stats import linregress
import numpy as np
import matplotlib.pyplot as plt


plt.figure(figsize=(10, 8))

# Logistic Map data
x1 = bif_array_shift_logmap
y1 = scaling_list_avg_logmap
log_x1 = np.log10(x1)
log_y1 = np.log10(y1)
slope1, intercept1, *_ = linregress(log_x1, log_y1)
fit_y1 = 10**(intercept1 + slope1 * np.log10(x1))

# Reservoir Model data
x2 = bif_array_shift
y2 = scaling_list_avg
log_x2 = np.log10(x2)
log_y2 = np.log10(y2)
slope2, intercept2, *_ = linregress(log_x2, log_y2)
fit_y2 = 10**(intercept2 + slope2 * np.log10(x2))

# Original data points
plt.loglog(x1, y1, marker='X', markersize  = 10, linestyle='None',
           color='black', label='Logistic Map: Slope = -0.54')
plt.loglog(x2, y2, marker='o', linestyle='None',
           color='darkred', label= 'Reservoir Map: Slope = -0.56')

# Power-law fits
plt.loglog(x1, fit_y1, 'k--', lw=2)
           
plt.loglog(x2, fit_y2, color='red', ls='--', lw=2)

# Labels
plt.xlabel(r"$\varepsilon - \varepsilon^*$")
plt.ylabel(r"$<\tau>$")

# Axis limits with padding
x_all = np.concatenate([x1, x2])
y_all = np.concatenate([y1, y2])
plt.xlim(min(x_all) * 0.9, max(x_all) * 1.1)
plt.ylim(min(y_all) * 0.9, max(y_all) * 1.1)


plt.grid(which='both', linestyle=':', linewidth=0.8, alpha=0.5)

plt.show()

